In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import os


In [25]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [5]:
DATA_DIR = '/kaggle/input/sagittal/sliced_images_sagittal'  
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

In [8]:
from torch.utils.data import random_split


In [9]:
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

total_size = len(dataset)
train_size = int(total_size * train_ratio)
val_size = int(total_size * val_ratio)
test_size = total_size - train_size - val_size 

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)


In [10]:
import json
class_to_idx = dataset.class_to_idx
with open('classes.json', 'w') as f:
    json.dump(class_to_idx, f)


In [11]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [13]:
import torch
import torch.nn as nn
from torchvision import models

model = models.densenet121(pretrained=True)

num_classes = 5  
model.classifier = nn.Linear(model.classifier.in_features, num_classes)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-4)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 159MB/s] 


In [16]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)


In [18]:

model = models.densenet121(pretrained=True)

for param in model.features.parameters():
    param.requires_grad = False

num_classes = 5
model.classifier = nn.Linear(model.classifier.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-4)


In [19]:
import time

total_epochs = 20 
start_epoch = 0   

for epoch in range(start_epoch, total_epochs):
    epoch_start = time.time()

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels).item()
        train_total += labels.size(0)

    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total * 100

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total * 100

    epoch_end = time.time()
    epoch_duration = epoch_end - epoch_start

   
    m, s = divmod(epoch_duration, 60)
    h, m = divmod(m, 60)

    print(f"Epoch {epoch+1}/{total_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
          f"Time: {int(h)}h {int(m)}m {int(s)}s")


Epoch 1/20 | Train Loss: 1.3570 | Train Acc: 44.27% | Val Loss: 1.2893 | Val Acc: 50.18% | Time: 0h 3m 51s
Epoch 2/20 | Train Loss: 1.2612 | Train Acc: 51.84% | Val Loss: 1.2241 | Val Acc: 53.83% | Time: 0h 2m 55s
Epoch 3/20 | Train Loss: 1.2084 | Train Acc: 53.91% | Val Loss: 1.1847 | Val Acc: 55.23% | Time: 0h 2m 52s
Epoch 4/20 | Train Loss: 1.1717 | Train Acc: 55.42% | Val Loss: 1.1460 | Val Acc: 56.32% | Time: 0h 2m 56s
Epoch 5/20 | Train Loss: 1.1410 | Train Acc: 56.27% | Val Loss: 1.1275 | Val Acc: 57.26% | Time: 0h 3m 0s
Epoch 6/20 | Train Loss: 1.1180 | Train Acc: 57.08% | Val Loss: 1.1022 | Val Acc: 57.81% | Time: 0h 2m 57s
Epoch 7/20 | Train Loss: 1.0987 | Train Acc: 57.57% | Val Loss: 1.0868 | Val Acc: 58.09% | Time: 0h 2m 56s
Epoch 8/20 | Train Loss: 1.0839 | Train Acc: 58.07% | Val Loss: 1.0788 | Val Acc: 58.14% | Time: 0h 2m 55s
Epoch 9/20 | Train Loss: 1.0679 | Train Acc: 58.76% | Val Loss: 1.0590 | Val Acc: 59.15% | Time: 0h 2m 56s
Epoch 10/20 | Train Loss: 1.0555 | Tra

In [20]:
model.eval()  
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")


Test Loss: 87.4451, Test Accuracy: 62.51%
